# Reading answer scripts with a VLMProduces one Markdown file per input image, named identically, ready toscore with `modules/06_evaluation/src/ocr_bench.py`.**Before uploading:** cover pages (`page_01`) carry names, USNs andmarks. They are already excluded by `07_reconstruct`, but check your zip.**Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

In [ ]:
!pip -q install "transformers>=4.49" accelerate qwen-vl-utils bitsandbytes

## 3. Upload your pagesZip the images first. Filenames become the output names, so use thepage ids the benchmark expects, e.g. `s06_c1_p05.png`.

In [ ]:
import zipfile, pathlibfrom google.colab import filesup = files.upload()                 # choose your .zipname = next(iter(up))IN = pathlib.Path('/content/pages'); IN.mkdir(exist_ok=True)with zipfile.ZipFile(name) as z:    z.extractall(IN)imgs = sorted(p for p in IN.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})print(f'{len(imgs)} image(s)')for p in imgs[:10]: print('  ', p.name)

## 4. Load the model`3B` is comfortable on a T4 and fast. `7B` is more accurate but needs4-bit to fit — try 3B first and only move up if the score demands it.

In [ ]:
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"   # or "Qwen/Qwen2.5-VL-7B-Instruct"FOUR_BIT = False                         # set True for the 7B on a T4import torchfrom transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessorkw = dict(torch_dtype=torch.bfloat16, device_map="auto")if FOUR_BIT:    from transformers import BitsAndBytesConfig    kw["quantization_config"] = BitsAndBytesConfig(        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, **kw)processor = AutoProcessor.from_pretrained(MODEL)print('loaded', MODEL)

## 5. The prompt

In [ ]:
PROMPT = """Transcribe this handwritten exam answer page to Markdown.Rules:- Transcribe EXACTLY what is written. Do not correct spelling, grammar,  arithmetic or factual errors. Student mistakes are the data.- If you cannot read something, write [?] instead of guessing a  plausible word. Never invent text.- Question numbers are written in the left margin (1, 2a, 2b, 2c, 3a,  3b, 4a, 4b). Emit each as a heading: ### 2a)- Sub-parts (i, ii, iii ... or a, b, c ...) become: #### i)- Mathematics: inline LaTeX between $ ... $- Tables: a Markdown table.- Diagrams, graphs, figures, flowcharts: do NOT describe them in prose.  Emit exactly this line and nothing else for them: ![diagram]- Struck-out or cancelled text: wrap in ~~ ~~- Keep the line breaks as written.Output only the Markdown. No commentary, no preamble."""print(PROMPT)

## 6. Read every page`min_pixels` / `max_pixels` matter: too small and the handwriting isunreadable, too large and a T4 runs out of memory on a 1598x2177 scan.

In [ ]:
import time, pathlibfrom PIL import ImageOUT = pathlib.Path('/content/markdown'); OUT.mkdir(exist_ok=True)MIN_PX, MAX_PX = 512*28*28, 1280*28*28def read(path):    image = Image.open(path).convert('RGB')    messages = [{"role": "user", "content": [        {"type": "image", "image": image,         "min_pixels": MIN_PX, "max_pixels": MAX_PX},        {"type": "text", "text": PROMPT}]}]    text = processor.apply_chat_template(        messages, tokenize=False, add_generation_prompt=True)    inputs = processor(text=[text], images=[image],                       return_tensors="pt").to(model.device)    with torch.no_grad():        out = model.generate(**inputs, max_new_tokens=1536, do_sample=False)    trimmed = out[0][inputs.input_ids.shape[1]:]    return processor.decode(trimmed, skip_special_tokens=True).strip()t0 = time.time()for i, p in enumerate(imgs, 1):    started = time.time()    try:        body = read(p)    except Exception as e:        body = ''        print(f'  {p.name}: FAILED {e}')    (OUT / (p.stem + '.md')).write_text(body, encoding='utf-8')    print(f'[{i}/{len(imgs)}] {p.stem}  {len(body)} chars  '          f'{time.time()-started:.0f}s', flush=True)print(f'\nTotal {time.time()-t0:.0f}s')

## 7. Spot-check one before downloading everything

In [ ]:
print((OUT / (imgs[0].stem + '.md')).read_text(encoding='utf-8')[:1500])

## 8. DownloadUnzip into `modules/06_evaluation/predictions/qwen_3b/`, then locally:```python modules/06_evaluation/src/ocr_bench.py --engine qwen_3b --verbose```That scores it against the same pages as the 0.573 CER baseline.

In [ ]:
import shutilfrom google.colab import filesshutil.make_archive('/content/markdown_out', 'zip', OUT)files.download('/content/markdown_out.zip')

## What to look for- **`[?]` markers** — the model admitting it cannot read. Good; that is  the behaviour TrOCR lacked.- **Fluent text that is not on the page** — fabrication. Compare a  couple of outputs against the images by eye before trusting the CER.- **`![diagram]`** where a figure is, rather than a prose description  of it.- **Question headings** (`### 2a)`) — these are what the reconstruction  step groups on, so they matter more than the prose around them.